## Experimenting with BERT

In [1]:
!pip install torch

In [2]:
!pip install scikit-learn

In [3]:
!pip install transformers datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 480.6/480.6 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 16.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.1/194.1 kB 18.5 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2024.10.0
    Uninstalling fsspec-2024.10.0:
      Successfully uninstalled fsspec-2024.10.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2024.10.0 requires fsspec==2024.10.0, but you have fsspec 2024.9.0 which is incompatible.


In [4]:
import os
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
import shutil

pos_dir = "/content/drive/My Drive/reviews_data/pos"
neg_dir = "/content/drive/My Drive/reviews_data/neg"

# Copy directories to local storage
shutil.copytree(pos_dir, '/data/pos_reviews')
shutil.copytree(neg_dir, '/data/neg_reviews')

'/data/neg_reviews'

In [6]:
# Update directories to local paths
pos_dir = '/data/pos_reviews'
neg_dir = '/data/neg_reviews'

def read_reviews(dir, label):
    reviews = []
    for filename in os.listdir(dir):
        file_path = os.path.join(dir, filename)
        with open(file_path, 'r') as file:
            reviews.append({"text": file.read(), "label": label})
    return reviews

pos_revs = read_reviews(pos_dir, 1)
neg_revs = read_reviews(neg_dir, 0)

print(f"{len(pos_revs)} positive reviews")
print(f"{len(neg_revs)} positive reviews")

2000 positive reviews
2000 positive reviews


In [18]:
from datasets import Dataset, DatasetDict

# Load data into Dataset object
all_revs = pos_revs + neg_revs
dataset = Dataset.from_list(all_revs)
print(dataset)

# Split dataset
first_split = dataset.train_test_split(test_size=0.3, seed=41)
second_split = first_split["test"].train_test_split(test_size=0.5, seed=41)

dataset_dict = DatasetDict({
    "train": first_split["train"],
    "dev": second_split["train"],
    "test": second_split["test"]
})

print(dataset_dict)

Dataset({
    features: ['text', 'label'],
    num_rows: 4000
})
DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 2800
    })
    dev: Dataset({
        features: ['text', 'label'],
        num_rows: 600
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 600
    })
})


# Uncased model

In [19]:
from transformers import BertTokenizer

# Uncased model
tokeniser = BertTokenizer.from_pretrained('bert-base-uncased')

def tokenise_function(examples):
    return tokeniser(examples["text"], truncation=True, padding=True)

tokenised_datasets = dataset_dict.map(tokenise_function, batched=True)

Map:   0%|          | 0/2800 [00:00<?, ? examples/s]

Map:   0%|          | 0/600 [00:00<?, ? examples/s]

Map:   0%|          | 0/600 [00:00<?, ? examples/s]

In [9]:
print(tokenised_datasets.column_names)

{'train': ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'], 'dev': ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'], 'test': ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask']}


In [20]:
tokenised_datasets.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

train_dataset = tokenised_datasets["train"]
dev_dataset = tokenised_datasets["dev"]
test_dataset = tokenised_datasets["test"]

print(f"Train size: {len(train_dataset)}, Dev size: {len(dev_dataset)}, Test size: {len(test_dataset)}")

Train size: 2800, Dev size: 600, Test size: 600


In [21]:
# Train
from transformers import BertForSequenceClassification, Trainer, TrainingArguments
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import torch

model = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)

training_args = TrainingArguments(
    "test-trainer",
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=5,
    learning_rate=2e-5,
    weight_decay=0.01,
    report_to="none",
    logging_dir="./logs",
    logging_steps=200,
)

def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average="binary")
    acc = accuracy_score(labels, preds)
    return {"accuracy": acc, "precision": precision, "recall": recall, "f1": f1}

trainer = Trainer(
    model,
    training_args,
    train_dataset=train_dataset,
    eval_dataset=dev_dataset,
    compute_metrics=compute_metrics
)

trainer.train()

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Step,Training Loss
200,0.550100
400,0.471700
600,0.456500
800,0.378100
1000,0.229300
1200,0.232500
1400,0.261900
1600,0.084100
1800,0.120500
2000,0.086300


TrainOutput(global_step=3500, training_loss=0.17671129081132156, metrics={'train_runtime': 349.7215, 'train_samples_per_second': 40.032, 'train_steps_per_second': 10.008, 'total_flos': 3683554775040000.0, 'train_loss': 0.17671129081132156, 'epoch': 5.0})

In [22]:
validation_set_results = trainer.evaluate()
print("Validation set results:\n", validation_set_results)

Validation set results:
 {'eval_loss': 0.6928912401199341, 'eval_accuracy': 0.9033333333333333, 'eval_precision': 0.8848684210526315, 'eval_recall': 0.9212328767123288, 'eval_f1': 0.9026845637583892, 'eval_runtime': 4.5164, 'eval_samples_per_second': 132.848, 'eval_steps_per_second': 33.212, 'epoch': 5.0}


In [23]:
# Test
test_results = trainer.predict(test_dataset)

# Extract predictions and metrics
predictions = test_results.predictions.argmax(-1)  # Get predicted labels
metrics = test_results.metrics

# Print results
print("Test Metrics:", metrics)

Test Metrics: {'test_loss': 0.6985855102539062, 'test_accuracy': 0.895, 'test_precision': 0.8993288590604027, 'test_recall': 0.8903654485049833, 'test_f1': 0.8948247078464107, 'test_runtime': 4.6433, 'test_samples_per_second': 129.218, 'test_steps_per_second': 32.304}


# Cased model

In [13]:
from transformers import BertTokenizer

# Cased model
tokeniser = BertTokenizer.from_pretrained('bert-base-cased')

def tokenise_function(examples):
    return tokeniser(examples["text"], truncation=True, padding=True)

tokenised_datasets = dataset_dict.map(tokenise_function, batched=True)

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/436k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

Map:   0%|          | 0/2800 [00:00<?, ? examples/s]

Map:   0%|          | 0/600 [00:00<?, ? examples/s]

Map:   0%|          | 0/600 [00:00<?, ? examples/s]

In [14]:
tokenised_datasets.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

train_dataset = tokenised_datasets["train"]
dev_dataset = tokenised_datasets["dev"]
test_dataset = tokenised_datasets["test"]

print(f"Train size: {len(train_dataset)}, Dev size: {len(dev_dataset)}, Test size: {len(test_dataset)}")

Train size: 2800, Dev size: 600, Test size: 600


In [15]:
# Train
from transformers import BertForSequenceClassification, Trainer, TrainingArguments
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import torch

model = BertForSequenceClassification.from_pretrained("bert-base-cased", num_labels=2)

training_args = TrainingArguments(
    "test-trainer",
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=5,
    learning_rate=2e-5,
    weight_decay=0.01,
    report_to="none",
    logging_dir="./logs",
    logging_steps=200,
)

def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average="binary")
    acc = accuracy_score(labels, preds)
    return {"accuracy": acc, "precision": precision, "recall": recall, "f1": f1}

trainer = Trainer(
    model,
    training_args,
    train_dataset=train_dataset,
    eval_dataset=dev_dataset,
    compute_metrics=compute_metrics
)

trainer.train()

model.safetensors:   0%|          | 0.00/436M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Step,Training Loss
200,0.576400
400,0.467000
600,0.473600
800,0.421700
1000,0.324000
1200,0.259200
1400,0.338600
1600,0.133200
1800,0.141800
2000,0.149000


TrainOutput(global_step=3500, training_loss=0.20410678625372905, metrics={'train_runtime': 347.8048, 'train_samples_per_second': 40.252, 'train_steps_per_second': 10.063, 'total_flos': 3683554775040000.0, 'train_loss': 0.20410678625372905, 'epoch': 5.0})

In [16]:
validation_set_results = trainer.evaluate()
print("Validation set results:\n", validation_set_results)

Validation set results:
 {'eval_loss': 0.7722319960594177, 'eval_accuracy': 0.885, 'eval_precision': 0.8704318936877077, 'eval_recall': 0.8972602739726028, 'eval_f1': 0.8836424957841484, 'eval_runtime': 4.4672, 'eval_samples_per_second': 134.313, 'eval_steps_per_second': 33.578, 'epoch': 5.0}


In [ ]:
test_results = trainer.predict(test_dataset)

predictions = test_results.predictions.argmax(-1)
metrics = test_results.metrics

print("Test Metrics:", metrics)

Test Metrics: {'test_loss': 0.7902214527130127, 'test_accuracy': 0.885, 'test_precision': 0.8945578231292517, 'test_recall': 0.8737541528239202, 'test_f1': 0.8840336134453781, 'test_runtime': 4.7237, 'test_samples_per_second': 127.02, 'test_steps_per_second': 31.755}
